# 02 - Data Profiling & Cleaning

## Doel
Data kwaliteit controleren (profiling) en voorbereiden voor analyse.

## Dit notebook bevat 3 secties:
1. **Data Profiling** - Controle van kwaliteit, nullen, verdelingen, outliers
2. **Missing Data Strategie** - Hoe we omgaan met ontbrekende waarden
3. **Data Cleaning** - Omzettingen, filtering, mergen

In [ ]:
# Cel 1: Inladen van bibliotheken en instellen van paden

import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from pathlib import Path

from src.profiling import (
    profile_dataframe, check_nulls, detect_outliers_iqr,
    correlation_report, distribution_summary, validate_portfolio
)
from src.utils import (
    load_portfolio, calculate_pure_gold_weight,
    filter_bars, filter_jewelry
)
from src.data_loader import load_raw

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
sns.set_palette('husl')

RAW_DIR = Path('../data/raw')
PROC_DIR = Path('../data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)
print('Setup complete.')

---
## SECTIE A: Data Profiling

Verplicht voor de eindproef - toon aan dat je data kwaliteit begrijpt.

### A1. Portfolio Data Profiling

In [ ]:
# Cel 2: A1. Portfolio Data Profiling

portfolio = load_portfolio()
profile_dataframe(portfolio, 'PORTFOLIO.csv')

In [ ]:
# Cel 3: A1. Portfolio Data Profiling

# Null analyse
print('--- Null Analyse ---')
null_report = check_nulls(portfolio)
print(null_report)

In [ ]:
# Cel 4: A1. Portfolio Data Profiling

# Visualisatie van ontbrekende waarden
msno.matrix(portfolio, figsize=(12, 6))
plt.title('Missing Values Matrix - Portfolio')
plt.tight_layout()
plt.show()

In [ ]:
# Cel 5: A1. Portfolio Data Profiling

# Validatie
validation = validate_portfolio(portfolio)
print(f"Portfolio valid: {validation['valid']}")
if validation['issues']:
    for issue in validation['issues']:
        print(f"  - {issue}")

In [ ]:
# Cel 6: A1. Portfolio Data Profiling

# Verdeling van karat waarden
print('--- Karat Verdeling ---')
print(portfolio['karaat'].value_counts().sort_index())
print()

# Verdeling van type
print('--- Type Verdeling ---')
print(portfolio['type'].value_counts())
print()

# Verdeling van herkomst
print('--- Herkomst Verdeling ---')
print(portfolio['herkomst goud'].value_counts())

In [ ]:
# Cel 7: A1. Portfolio Data Profiling

# Outlier detectie op gram en aankoopprijs
print('--- Outlier Detectie: Gram ---')
gram_outliers = detect_outliers_iqr(portfolio['gram'])
print(f"Aantal outliers: {len(gram_outliers)}")
if len(gram_outliers) > 0:
    print(gram_outliers)

print('\n--- Outlier Detectie: Aankoopprijs ---')
prijs_outliers = detect_outliers_iqr(portfolio[portfolio['aankoopprijs'] > 0]['aankoopprijs'])
print(f"Aantal outliers (alleen > EUR 0): {len(prijs_outliers)}")
if len(prijs_outliers) > 0:
    print(prijs_outliers)

In [ ]:
# Cel 8: A1. Portfolio Data Profiling

# Box plots voor outliers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

portfolio.boxplot(column='gram', ax=axes[0])
axes[0].set_title('Verdeling Gram (met outliers)')
axes[0].set_ylabel('Gram')

bars = filter_bars(portfolio)
bars.boxplot(column='aankoopprijs', ax=axes[1])
axes[1].set_title('Verdeling Aankoopprijs (goudstaven)')
axes[1].set_ylabel('EUR')

plt.tight_layout()
plt.show()

### A2. Goudprijs Data Profiling

In [ ]:
# Cel 9: A2. Goudprijs Data Profiling

gold_macro = load_raw('gold_macro_merged.csv')
profile_dataframe(gold_macro, 'Gold + Macro Data')

In [ ]:
# Cel 10: A2. Goudprijs Data Profiling

# Correlatie matrix
corr = correlation_report(gold_macro)
print('--- Correlatie Matrix ---')
print(corr.round(3))

In [ ]:
# Cel 11: A2. Goudprijs Data Profiling

# Correlatie heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdYlGn', center=0, fmt='.2f',
            square=True, linewidths=0.5)
plt.title('Correlatie Matrix - Goud & Macro Indicatoren')
plt.tight_layout()
plt.show()

In [ ]:
# Cel 12: A2. Goudprijs Data Profiling

# Distributie van dagelijkse goudrendementen
gold_macro['daily_return'] = gold_macro['gold_usd_oz'].pct_change()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gold_macro['daily_return'].hist(bins=50, ax=axes[0], alpha=0.7, color='gold')
axes[0].set_title('Verdeling Dagelijkse Rendementen')
axes[0].set_xlabel('Dagelijks rendement')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.7)

stats = distribution_summary(gold_macro['daily_return'].dropna())
axes[1].barh(list(stats.keys()), list(stats.values()), color='steelblue')
axes[1].set_title('Distributie Statistieken')

plt.tight_layout()
plt.show()

print('--- Rendement Statistieken ---')
for k, v in stats.items():
    print(f"  {k}: {v:.6f}")

---
## SECTIE B: Missing Data Strategie

| Ontbrekend Type | Afhandeling | Reden |
|----------------|-------------|-------|
| Goudprijs gaps (weekends/feestdagen) | Forward-fill | Goudmarkt gesloten; laatste bekende prijs is geldig |
| Macro data gaps | Interpolate | Macro data beweegt langzaam |
| EUR/USD gaps | Forward-fill | Zelfde reden als goud |
| Portfolio ontbrekende prijzen (sieraden) | Zet op 0 + vlagkolom | Familie erfstukken; geen marktwaarde om te volgen |

In [ ]:
# Cel 13: SECTIE B: Missing Data Strategie

# Missing data in gold_macro
print('--- Missing Data: Gold + Macro ---')
print(check_nulls(gold_macro))
print(f'\nTotaal rijen: {len(gold_macro)}')

In [ ]:
# Cel 14: SECTIE B: Missing Data Strategie

# Missing data in portfolio
print('--- Missing Data: Portfolio ---')
print(check_nulls(portfolio))

---
## SECTIE C: Data Cleaning

### C1. Portfolio Cleaning

In [ ]:
# Cel 15: C1. Portfolio Cleaning

# Zuiver goud gewicht berekenen
portfolio_clean = calculate_pure_gold_weight(portfolio)

# Kolom toevoegen voor ontbrekende prijzen (sieraden)
portfolio_clean['prijs_ontbreekt'] = portfolio_clean['aankoopprijs'] == 0

print(f"Portfolio: {len(portfolio_clean)} items")
print(f"  Staven: {len(filter_bars(portfolio_clean))}")
print(f"  Sieraden: {len(filter_jewelry(portfolio_clean))}")
print(f"  Items met ontbrekende prijs: {portfolio_clean['prijs_ontbreekt'].sum()}")

portfolio_clean.head(10)

In [ ]:
# Cel 16: C1. Portfolio Cleaning

# Samenvatting per type
print('--- Samenvatting per Type ---')
for t in ['staaf', 'juweel']:
    subset = portfolio_clean[portfolio_clean['type'] == t]
    print(f"\n{t.upper()}:")
    print(f"  Aantal: {len(subset)}")
    print(f"  Totaal gewicht: {subset['gram'].sum():.2f} g")
    print(f"  Zuiver goud: {subset['zuiver_goud_gram'].sum():.2f} g")
    print(f"  Totaal geinvesteerd: EUR {subset['aankoopprijs'].sum():.2f}")

### C2. Gold Price Data Cleaning

In [ ]:
# Cel 17: C2. Gold Price Data Cleaning

# Forward-fill voor prijzen
gold_clean = gold_macro.copy()

print(f'Voor cleaning: {gold_clean.shape[0]} rijen, {gold_clean.isnull().sum().sum()} ontbrekende waarden')

# Forward-fill voor prijs kolommen
price_cols = [c for c in gold_clean.columns if c != 'daily_return']
gold_clean[price_cols] = gold_clean[price_cols].ffill()

# Resterende NaNs aan het begin verwijderen
gold_clean = gold_clean.dropna(subset=['gold_usd_oz'])

# Daily return herberekenen
gold_clean['daily_return'] = gold_clean['gold_usd_oz'].pct_change()

print(f'Na cleaning: {gold_clean.shape[0]} rijen, {gold_clean.isnull().sum().sum()} ontbrekende waarden')
gold_clean.head(10)

In [ ]:
# Cel 18: C2. Gold Price Data Cleaning

# EUR conversie kolommen toevoegen (indien niet aanwezig)
TROY_OZ_TO_GRAM = 31.1035

if 'gold_eur_oz' not in gold_clean.columns:
    gold_clean['gold_eur_oz'] = gold_clean['gold_usd_oz'] * gold_clean['eur_usd']
    gold_clean['gold_eur_gram'] = gold_clean['gold_eur_oz'] / TROY_OZ_TO_GRAM
    gold_clean['gold_10g_eur'] = gold_clean['gold_eur_gram'] * 10
    print('EUR kolommen toegevoegd')
else:
    print('EUR kolommen reeds aanwezig')

gold_clean[['gold_usd_oz', 'eur_usd', 'gold_eur_gram', 'gold_10g_eur']].tail(10)

### C3. Portfolio samenvoegen met spot prijzen

In [ ]:
# Cel 19: C3. Portfolio samenvoegen met spot prijzen

# Voor elke aankoop de spot prijs op die dag vinden
bars = filter_bars(portfolio_clean).copy()

spot_prices_at_purchase = []
for idx, row in bars.iterrows():
    aankoop_datum = row['datum_aankoop']
    # Zoek de dichtstbijzijnde datum in de goud data
    if 'gold_eur_gram' in gold_clean.columns:
        idx_nearest = gold_clean.index.get_indexer([aankoop_datum], method='nearest')[0]
        spot_prijs = gold_clean.iloc[idx_nearest]['gold_eur_gram']
    else:
        spot_prijs = np.nan
    spot_prices_at_purchase.append(spot_prijs)

bars['spot_prijs_aankoop'] = spot_prices_at_purchase
bars['spot_waarde_gram'] = bars['gram'] * bars['spot_prijs_aankoop']
bars['verschil_vs_spot'] = (bars['aankoopprijs'] / bars['gram']) - bars['spot_prijs_aankoop']

print('--- Goudstaven met Spot Prijs Vergelijking ---')
bars[['datum_aankoop', 'omschrijving', 'gram', 'aankoopprijs', 'spot_prijs_aankoop', 'spot_waarde_gram', 'verschil_vs_spot']].to_string()

In [ ]:
# Cel 20: C3. Portfolio samenvoegen met spot prijzen

# Gemiddelde prijs per gram vs spot
bars['prijs_per_gram'] = bars['aankoopprijs'] / bars['gram']

print(f"Gemiddelde aankoopprijs per gram: EUR {bars['prijs_per_gram'].mean():.2f}")
print(f"Gemiddelde spot prijs per gram: EUR {bars['spot_prijs_aankoop'].mean():.2f}")
print(f"Verschil: EUR {bars['verschil_vs_spot'].mean():.2f} per gram (premium)")

### C4. Schone data opslaan

In [ ]:
# Cel 21: C4. Schone data opslaan

# Portfolio opslaan
portfolio_clean.to_csv(PROC_DIR / 'portfolio_cleaned.csv', index=False)
print(f'Opgeslagen: portfolio_cleaned.csv ({len(portfolio_clean)} rijen)')

# Gold macro opslaan
gold_clean.to_csv(PROC_DIR / 'gold_with_macro.csv')
print(f'Opgeslagen: gold_with_macro.csv ({len(gold_clean)} rijen)')

# Staven met spot prijs opslaan
bars.to_csv(PROC_DIR / 'bars_with_spot.csv', index=False)
print(f'Opgeslagen: bars_with_spot.csv ({len(bars)} rijen)')

print('\nAlle schone data opgeslagen in data/processed/')

---
## Samenvatting

| Stap | Resultaat |
|------|----------|
| Profiling | Compleet - nullen, outliers, distributies, correlaties gecheckt |
| Missing Data | Forward-fill voor prijzen, interpolatie voor macro |
| Cleaning | Zuiver goud berekend, EUR conversie, spot prijs vergelijking |
| Validatie | Portfolio gevalideerd op types, karat waarden, negatieve waarden |

**Volgende stap:** Notebook 03 - Exploratory Data Analysis